# LTP Features + Gradient Boosting — Image Classification Pipeline

**Feature descriptor:** Local Ternary Pattern (LTP) &nbsp;|&nbsp; **Classifier:** Gradient Boosting Classifier &nbsp;|&nbsp; **Validation:** 5-fold cross-validation

---

This notebook implements an end-to-end image-classification experiment:

1. **Feature Extraction** — the Local Ternary Pattern descriptor (ternary threshold ±5, 8-neighborhood) yields concatenated positive/negative 256-bin histograms (512-dimensional feature vector) per image.
2. **Data Loading** — images are read fold-by-fold from a directory structure of `train` / `test` splits.
3. **Model Training & Evaluation** — a **Gradient Boosting classifier** — `GradientBoostingClassifier(random_state=42)` is trained on each fold, across six image resolutions, and evaluated with accuracy, precision, recall, and F1 (weighted & macro).
4. **Results** — per-fold metrics, 5-fold averages, and confusion matrices are reported; results are exported to CSV.

**Outputs**

| File | Contents |
|---|---|
| `LTP_GradientBoosting_5Fold_All_Results.csv` | Metrics for every fold x image size |
| `LTP_GradientBoosting_5Fold_Average_Results.csv` | Metrics averaged over the 5 folds |


## 1&nbsp;&nbsp;Imports & Logging

Standard scientific-Python stack: OpenCV and scikit-image for image processing, scikit-learn for the classifier and metrics, and matplotlib/seaborn for visualization. Logging is configured to report progress and any per-image failures without halting the experiment.


In [ ]:
import os
import logging
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from joblib import Parallel, delayed

from skimage import io, color
from skimage.transform import resize

from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)


# ======================================================
# LOGGING
# ======================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)




## 2&nbsp;&nbsp;LTP Feature Extraction

Two functions implement the Local Ternary Pattern descriptor:

- **`compute_ltp(image, threshold=5)`** — for every interior pixel, compares each of its 8 neighbors against the center using a ternary rule: differences above `+threshold` set a bit in a *positive* code, differences below `-threshold` set a bit in a *negative* code. The two code maps are summarized as 256-bin normalized histograms and concatenated into a **512-dimensional feature vector**.
- **`extract_ltp_features(image, target_size, threshold=5)`** — converts the image to grayscale (8-bit), resizes it to `target_size`, and computes the LTP histogram.

The ternary threshold (±5) makes LTP more robust to noise than plain LBP. Extraction failures are logged and the image is skipped.


In [ ]:
# ======================================================
# LTP FEATURE EXTRACTION
# ======================================================

def compute_ltp(image, threshold=5):
    """
    Compute Local Ternary Pattern (LTP).
    Returns concatenated positive and negative histograms.
    """

    rows, cols = image.shape

    positive_codes = np.zeros((rows - 2, cols - 2), dtype=np.uint8)
    negative_codes = np.zeros((rows - 2, cols - 2), dtype=np.uint8)

    offsets = [
        (-1, -1), (-1, 0), (-1, 1),
        (0, 1),
        (1, 1), (1, 0), (1, -1),
        (0, -1)
    ]

    for i in range(1, rows - 1):
        for j in range(1, cols - 1):

            center = image[i, j]

            pos_code = 0
            neg_code = 0

            for idx, (dx, dy) in enumerate(offsets):
                neighbor = image[i + dx, j + dy]

                diff = int(neighbor) - int(center)

                if diff > threshold:
                    pos_code |= (1 << idx)

                elif diff < -threshold:
                    neg_code |= (1 << idx)

            positive_codes[i - 1, j - 1] = pos_code
            negative_codes[i - 1, j - 1] = neg_code

    pos_hist, _ = np.histogram(
        positive_codes.ravel(),
        bins=256,
        range=(0, 256),
        density=True
    )

    neg_hist, _ = np.histogram(
        negative_codes.ravel(),
        bins=256,
        range=(0, 256),
        density=True
    )

    return np.hstack([pos_hist, neg_hist])


def extract_ltp_features(image, target_size, threshold=5):
    try:
        if image.ndim == 3:
            gray_image = color.rgb2gray(image)
            gray_image = (gray_image * 255).astype(np.uint8)
        else:
            gray_image = image.astype(np.uint8)

        resized_image = resize(
            gray_image,
            target_size,
            anti_aliasing=True,
            preserve_range=True
        ).astype(np.uint8)

        features = compute_ltp(resized_image, threshold)

        return np.array(features, dtype=np.float32)

    except Exception as e:
        logging.error(f"LTP feature extraction failed: {e}")
        return None




## 3&nbsp;&nbsp;Data Loading

`load_data(directory, target_size)` discovers class sub-directories, extracts LTP features for every image (classes processed in parallel across all CPU cores), and returns the feature matrix `X`, integer-encoded labels `y`, and the ordered list of class names.


In [ ]:
# ======================================================
# DATA LOADING FUNCTION
# ======================================================

def load_data(directory, target_size):

    class_names = sorted([
        d for d in os.listdir(directory)
        if os.path.isdir(os.path.join(directory, d))
    ])

    def process_class(class_name):
        class_path = os.path.join(directory, class_name)

        feature_list = []
        label_list = []

        for file_name in tqdm(
            os.listdir(class_path),
            desc=f"Processing {class_name}"
        ):

            if file_name.startswith("."):
                continue

            image_path = os.path.join(class_path, file_name)

            try:
                image = io.imread(image_path)
            except Exception as e:
                logging.warning(f"Cannot read {image_path}: {e}")
                continue

            features = extract_ltp_features(image, target_size)

            if features is not None:
                feature_list.append(features)
                label_list.append(class_name)

        return feature_list, label_list

    results = Parallel(n_jobs=-1)(
        delayed(process_class)(cls)
        for cls in tqdm(class_names, desc="Loading Classes")
    )

    X = []
    y = []

    for features, labels in results:
        X.extend(features)
        y.extend(labels)

    y = np.array([class_names.index(label) for label in y])

    return np.array(X), y, class_names




## 4&nbsp;&nbsp;Experiment Configuration

Defines the dataset location, the five cross-validation folds, and the six image resolutions to be evaluated. `all_results` accumulates the metrics from every (image size, fold) run.


In [ ]:
# ======================================================
# 5-FOLD SETTINGS
# ======================================================

baseDir = r"E:\THUSHAR\DATASET\Croped_5Fold"

# For Google Colab use:
# baseDir = "/content/drive/MyDrive/Croped_5Fold"

folds = [
    "fold_1",
    "fold_2",
    "fold_3",
    "fold_4",
    "fold_5"
]


# ======================================================
# DIFFERENT IMAGE SIZES
# ======================================================

sizes = [
    (8, 8),
    (16, 16),
    (32, 32),
    (64, 64),
    (128, 128),
    (196, 210)
]


# ======================================================
# STORE ALL RESULTS
# ======================================================

all_results = []




## 5&nbsp;&nbsp;Training & Evaluation — Main Experiment Loop

For every image size and every fold:

1. Load the fold's train and test sets.
2. Train a **Gradient Boosting classifier** — `GradientBoostingClassifier(random_state=42)`.
3. Predict on the test set and compute **accuracy**, **precision / recall / F1** (weighted and macro).
4. Append the metrics to `all_results` and plot the **confusion matrix**.


In [ ]:
# ======================================================
# MAIN EXPERIMENT LOOP
# ======================================================

for size in sizes:

    print("\n========================================")
    print(f"PROCESSING IMAGE SIZE: {size[0]}x{size[1]}")
    print("========================================")

    for fold in folds:

        print(f"\n########## {fold} ##########")

        mainDir = os.path.join(baseDir, fold)

        # --------------------------------------------------
        # LOAD DATA
        # --------------------------------------------------

        X_train, y_train, class_names = load_data(
            os.path.join(mainDir, "train"),
            size
        )

        X_test, y_test, _ = load_data(
            os.path.join(mainDir, "test"),
            size
        )

        print("Train shape:", X_train.shape)
        print("Test shape :", X_test.shape)

        # --------------------------------------------------
        # GRADIENT BOOSTING CLASSIFIER
        # --------------------------------------------------

        clf = GradientBoostingClassifier(
            random_state=42
        )

        # --------------------------------------------------
        # TRAIN
        # --------------------------------------------------

        print("\nTraining Gradient Boosting ...")

        clf.fit(X_train, y_train)

        # --------------------------------------------------
        # PREDICT
        # --------------------------------------------------

        y_pred = clf.predict(X_test)

        # ==================================================
        # METRICS
        # ==================================================

        accuracy = accuracy_score(y_test, y_pred)

        precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0
        )

        precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(
            y_test,
            y_pred,
            average="macro",
            zero_division=0
        )

        # ==================================================
        # DISPLAY METRICS
        # ==================================================

        print(f"Accuracy            : {accuracy:.4f}")
        print(f"F1 Weighted         : {f1_w:.4f}")
        print(f"F1 Macro            : {f1_m:.4f}")

        # ==================================================
        # SAVE RESULTS
        # ==================================================

        all_results.append({
            "Fold": fold,
            "Feature": "LTP",
            "Classifier": "GradientBoosting",
            "Image_Size": f"{size[0]}x{size[1]}",
            "Accuracy": accuracy,
            "Precision_Weighted": precision_w,
            "Recall_Weighted": recall_w,
            "F1_Weighted": f1_w,
            "Precision_Macro": precision_m,
            "Recall_Macro": recall_m,
            "F1_Macro": f1_m
        })

        # --------------------------------------------------
        # CONFUSION MATRIX
        # --------------------------------------------------

        cm = confusion_matrix(y_test, y_pred)

        plt.figure(figsize=(10, 8))

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names
        )

        plt.title(
            f"LTP + Gradient Boosting\n{fold} ({size[0]}x{size[1]})"
        )

        plt.xlabel("Predicted Class")
        plt.ylabel("Actual Class")
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()




## 6&nbsp;&nbsp;Results Aggregation & Export

Fold-wise metrics are saved to CSV, averaged across the five folds per image size, exported, and displayed sorted by image size and accuracy.


In [ ]:
# ======================================================
# SAVE FOLD-WISE RESULTS
# ======================================================

results_df = pd.DataFrame(all_results)

results_df.to_csv(
    "LTP_GradientBoosting_5Fold_All_Results.csv",
    index=False
)

print(
    "\nFold-wise results saved: "
    "LTP_GradientBoosting_5Fold_All_Results.csv"
)


# ======================================================
# COMPUTE AVERAGE 5-FOLD RESULTS
# ======================================================

average_df = results_df.groupby(
    ["Feature", "Classifier", "Image_Size"]
).agg({
    "Accuracy": "mean",
    "Precision_Weighted": "mean",
    "Recall_Weighted": "mean",
    "F1_Weighted": "mean",
    "Precision_Macro": "mean",
    "Recall_Macro": "mean",
    "F1_Macro": "mean"
}).reset_index()

average_df.to_csv(
    "LTP_GradientBoosting_5Fold_Average_Results.csv",
    index=False
)

print(
    "Average results saved: "
    "LTP_GradientBoosting_5Fold_Average_Results.csv"
)


# ======================================================
# DISPLAY FINAL RESULTS
# ======================================================

print("\n========================================")
print("FINAL 5-FOLD AVERAGE RESULTS")
print("========================================")

print(
    average_df.sort_values(
        ["Image_Size", "Accuracy"],
        ascending=[True, False]
    )
)

print("\n========================================")
print("LTP + GRADIENT BOOSTING EXPERIMENTS COMPLETED SUCCESSFULLY")
print("========================================")